<a href="https://colab.research.google.com/github/anindyaroy/learnAgenticAI/blob/main/CrewAI_Advanced_Python_Codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Fitness Tracker with Crew AI
from crewai import Agent, Task, Crew, LLM
from crewai_tools import SerperDevTool
import os

# Set up API key
os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

# Initialize LLM for natural language processing
llm = LLM(model="gpt-4")

# User inputs
fitness_goal = "muscle gain"
nutrition_preference = "high protein"

historical_weight_data = ["87kg", "85kg", "88kg", "87kg", "86kg"]

# Fitness Tracker Agent
fitness_tracker_agent = Agent(
    llm=llm,
    role="Fitness Tracker",
    backstory="An AI-powered fitness tracker agent that helps users set goals, track progress, and recommend personalized workout plans.",
    goal="Set fitness goals, track user progress, and provide personalized recommendations based on user preferences.",
    verbose=True,
)

recommendation_agent = Agent(
    llm=llm,
    role="Recommendation Agent",
    backstory="An AI agent that fetches personalized fitness and nutrition recommendations based on user goals and preferences.",
    goal="Search for fitness and nutrition information to provide recommendations that match the user's goals.",
    tools=[SerperDevTool()],
    verbose=True,
)

# Tasks

# 1. Set fitness goals (muscle gain) and track progress
set_and_track_goals = Task(
    description=f"Set the user's fitness goal to {fitness_goal} and track their progress with a "
                f"personalized plan, also take into account the last weight measurements: {historical_weight_data}.",
    expected_output="A fitness plan for muscle gain and progress tracking setup.",
    agent=fitness_tracker_agent,
)

# 2. Fetch personalized fitness and nutrition recommendations based on user preferences (high protein diet)
fetch_fitness_recommendations = Task(
    description=f"Fetch fitness recommendations for goal {fitness_goal} and nutrition preference {nutrition_preference}.",
    expected_output="Personalized fitness and nutrition recommendations based on the user's goal and dietary preference.",
    agent=recommendation_agent,
)

# 3. Provide step-by-step workout plan
provide_workout_plan = Task(
    description="Provide a step-by-step workout plan to help the user achieve their fitness goal.",
    expected_output="A personalized workout plan to meet the muscle gain goal.",
    agent=fitness_tracker_agent,
)

# Crew setup
crew = Crew(
    agents=[fitness_tracker_agent, recommendation_agent],
    tasks=[set_and_track_goals, fetch_fitness_recommendations, provide_workout_plan],
    planning=True,
)

# Start the process
crew.kickoff()

In [ ]:
### Images generation .py

from crewai import Crew, Agent, Task, LLM
from crewai_tools import DallETool
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type, Any
import requests
import os

# Set up API key
os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

llm = LLM(model="gpt-4")

class DownloadImageInput(BaseModel):
    image_url: str = Field(..., description="The URL of the image to download")

class DownloadImageTool(BaseTool):
    name: str = "Image Downloader Tool"
    description: str = "Downloads image from url"
    args_schema: Type[BaseModel] = DownloadImageInput

    def _run(self, image_url: str) -> str:
        filename = f"{os.path.basename(image_url)}.png"

        try:
            response = requests.get(image_url, stream=True)
            response.raise_for_status()
        except requests.RequestException as e:
            return f"Failed {e}"

        with open(filename, "wb") as image_file:
            for chunk in response.iter_content(chunk_size=8192):
                image_file.write(chunk)

        return f"Save as {filename}"


dalle_tool = DallETool(model="dall-e-3",
                       size="1024x1024",
                       quality="standard",
                       n=1)

prompt_improver_agent = Agent(
    llm=llm,
    role="Prompt improver",
    backstory="Ai agent that enhances prompts to be more detailed",
    goal="Improve text prompts for optimal image generation",
    verbose=True
)

image_generator_agent = Agent(
    llm=llm,
    role="Image generator",
    backstory="An AI agent that generates images based on detailed prompts, using DALLE and save the image in the current directory",
    goal="Generate images from enhanced prompts using DALLE and save them locally",
    tools=[dalle_tool, DownloadImageTool()],
    verbose=True
)

enhance_prompt_task = Task(
    description="Improve this prompt ({initial_prompt}) to make it more descriptive and detailed for image generation.",
    expected_output='"image_description": "Enhanced Prompt here',
    agent=prompt_improver_agent
)

generate_image_task = Task(
    description="Generate an image from the enhanced prompt using DALLE",
    expected_output="image_url: URL of the generated image",
    agent=image_generator_agent
)

download_image_task = Task(
    description="Download the generated image from the provided URL",
    expected_output="Image downloaded and saved locally",
    agent=image_generator_agent
)

crew = Crew(
    agents=[prompt_improver_agent, image_generator_agent],
    tasks=[enhance_prompt_task, generate_image_task, download_image_task]
)

crew.kickoff(inputs={'initial_prompt': 'A futuristic cityscape at sunset'})

In [ ]:
### Book Recommendation .py

from crewai import Agent, Crew, Task, Process
from mem0 import MemoryClient
import os

# Set up API key
os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

def get_recommendation(user_id: str, user_preferences: str) -> str:
    recommendation_agent = Agent(
        role="Book Recommendation",
        backstory="Ai-powered book advisor who suggests books based on user preferences",
        goal="Provide personalised book recommendations.",
        verbose=True,
    )

    suggest_book_task = Task(
        description=f"Suggest three personalised book recommendations based on user preferences: {user_preferences}.",
        expected_output="A list of three book titles with brief description",
        agent=recommendation_agent
    )

    crew = Crew(
        agents=[recommendation_agent],
        tasks=[suggest_book_task],
        verbose=True,
        process=Process.sequential,
        memory=True,
        memory_config={
            "provider": "mem0",
            "config": {"user_id": user_id, "api_key": "m0-R5BIOR7mMfo8LViJKAAdt6eF6FwTrKGzeLxRO7F5"},
        }
    )

    result = crew.kickoff()

    print("Final Recommendation:")
    print(result.raw)

    return result.raw


def store_user_recommendation(client: MemoryClient, user_id: str, conversation: list):
    client.add(conversation, user_id=user_id)


if __name__ == '__main__':
    client = MemoryClient(api_key="m0-R5BIOR7mMfo8LViJKAAdt6eF6FwTrKGzeLxRO7F5")

    user_id = input("Enter user ID: ").strip()
    user_preferences = input("Enter preferences: ").strip()

    result = get_recommendation(user_id, user_preferences)

    conversation = [
        {
            "role": "user",
            "content": f"Recommendation for {user_preferences}"
        },
        {
            "role": "assistant",
            "content": result
        }
    ]

    store_user_recommendation(client, user_id, conversation)
